# 03 — Peak detection and tentative isotope identification

This notebook demonstrates peak detection and first-pass isotope matching.

Important scientific note:

SIMS peak assignment is not unique. Atomic ions, molecular ions, hydrides, oxides,
clusters, and matrix effects can all contribute. This notebook performs **candidate
matching**, not definitive chemical identification.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd

from pymagsims import Spectrum
from pymagsims.isotopes import load_builtin_isotopes

DATA = Path("../data")
spec = Spectrum.from_main_analysis_file(DATA / "FPD_01_2604281458290.csv")
isotopes = load_builtin_isotopes()

## Detect peaks

In [ ]:
peaks = spec.find_peaks(
    prominence=100,
    distance=5,
)

display(peaks.head(20))
print(f"Detected peaks: {len(peaks)}")

## Assign isotope candidates

In [ ]:
assignments = spec.assign_peaks(
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=100,
    distance=5,
)

display(assignments.head(30))
print(f"Candidate assignments: {len(assignments)}")

## Annotated spectrum

In [ ]:
fig, ax, assignments = spec.plot_with_peaks(
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=100,
    distance=5,
    log_y=True,
    annotate=True,
)

## Compare measured spectrum with isotope positions for selected elements

In [ ]:
spec.plot_with_element_markers(
    isotope_table=isotopes,
    elements=["Si", "Ga", "O"],
    log_y=True,
    xlim=(10, 80),
    min_abundance=0.1,
);

## Store automatic peak bins

Save the automatically generated bins to disk so they can be reused in later workflows,
for example in the 3D raw image notebook.

The saved CSV contains one row per bin and should include at least:

- `label`
- `mass_min`
- `mass_max`

If channel limits are added later, they can also be stored as `ch_min` / `ch_max`.

In [ ]:
from pathlib import Path

BIN_DIR = Path("../data/bins")
BIN_DIR.mkdir(parents=True, exist_ok=True)

# Create bins from the current automatic assignments.
automatic_bins = spec.create_bins_from_assignments(
    assignments,
    width=0.3,
)

display(automatic_bins.head(20))

automatic_bin_file = BIN_DIR / "automatic_bins.csv"
automatic_bins.to_csv(automatic_bin_file, index=False)

print(f"Saved automatic bins to: {automatic_bin_file}")

In [ ]:
# Optional sanity check: reload the saved bins.
reloaded_bins = pd.read_csv(automatic_bin_file)
display(reloaded_bins.head())